In [1]:
import os
import time
from tqdm import tqdm

MODE = 'test'
X265 = '/home/zhaoy/x265/build/linux/x265'
QPs = list(range(22, 52, 5))
PRESET = 'medium'
DEBUG  = False

#### 1. encoding

In [2]:
def enc_x265(size, yuv_dir, rlt_root, x265=X265, qps=QPs, preset=PRESET, debug=DEBUG):
    yuvs = [_ for _ in os.listdir(yuv_dir) if _.endswith('.yuv')]
    for yuv in tqdm(yuvs):
        yuv_path = os.path.join(yuv_dir, yuv)
        os.makedirs(f'{rlt_root}/log', exist_ok=True)
        os.makedirs(f'{rlt_root}/bin', exist_ok=True)
        os.makedirs(f'{rlt_root}/rec', exist_ok=True)
        
        for qp in qps:
            log_path = os.path.join(rlt_root, 'log', yuv.replace('.yuv', f'_qp{qp}_{PRESET}.csv'))
            bin_path = os.path.join(rlt_root, 'bin', yuv.replace('.yuv', f'_qp{qp}_{PRESET}.bin'))
            rec_path = os.path.join(rlt_root, 'rec', yuv.replace('.yuv', f'_qp{qp}_{PRESET}.yuv'))
            
            cmd = f'{x265} --input-res {size} --preset {preset} --qp {qp} --fps 30 --psnr --ssim --tune psnr --input {yuv_path} --output {bin_path} --recon {rec_path} --csv-log-level 2 --csv {log_path} --log-level info &'
            os.system(cmd)
            if DEBUG:
                break
        
        time.sleep(1)

In [3]:
''' Touch and Go '''
yuv_root = '/data/ssd/zhaoy/datasets/TouchandGoDataset-v2/dataset-comp'
yuv_dir  = os.path.join(yuv_root, MODE, 'video', 'yuv')
rlt_root = '/data/ssd/zhaoy/datasets/TouchandGoDataset-v2/compressed/x265'

# enc_x265(size='640x480', yuv_dir=yuv_dir, rlt_root=rlt_root)

In [4]:
''' Object Folder '''
yuv_root = '/data/ssd/zhaoy/datasets/ObjectFolder_1.0/dataset-comp'
yuv_dir  = os.path.join(yuv_root, MODE, 'video', 'yuv')
rlt_root = '/data/ssd/zhaoy/datasets/ObjectFolder_1.0/compressed/x265'

# enc_x265(size='160x120', yuv_dir=yuv_dir, rlt_root=rlt_root)

In [5]:
''' SSVTP '''
yuv_root = '/data/ssd/zhaoy/datasets/SSVTP/dataset-comp'
yuv_dir  = os.path.join(yuv_root, MODE, 'video', 'yuv')
rlt_root = '/data/ssd/zhaoy/datasets/SSVTP/compressed/x265'

# enc_x265(size='240x320', yuv_dir=yuv_dir, rlt_root=rlt_root)

In [6]:
''' YCB-Slide '''
yuv_root = '/data/ssd/zhaoy/datasets/YCB-Slide/dataset-comp'
yuv_dir  = os.path.join(yuv_root, MODE, 'video', 'yuv')
rlt_root = '/data/ssd/zhaoy/datasets/YCB-Slide/compressed/x265'

# enc_x265(size='240x320', yuv_dir=yuv_dir, rlt_root=rlt_root)

#### 2. get info

In [7]:
def extract_metrics(csv_path):
    with open(csv_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    # 找到 Summary 行
    for i, line in enumerate(lines):
        if line.startswith('Summary'):

            if i + 2 < len(lines):
                values = [x.strip() for x in lines[i+2].strip().split(',')]

                elapsed_time = float(values[2])
                bitrate = float(values[4])
                psnr = float(values[8])
                ssim = float(values[9])
                return [elapsed_time, bitrate, psnr, ssim]
            else:
                print("Summary value line not found.")
                return
    print("Summary not found.")

In [8]:
from msssim import calc_msssim_yuv

In [9]:
''' 1. Touch and Go '''
DATASET = 'TouchandGoDataset-v2'

import pandas as pd
# log_dir = f'/data/ssd/zhaoy/datasets/{DATASET}/compressed/x265/log'

# rows = []
# for log in os.listdir(log_dir):
#     if log.endswith('.csv'):
#         video_name, qp, preset = log.replace('.csv', '').split('_')
#         qp = int(qp.replace('qp', ''))
#         csv_path = os.path.join(log_dir, log)
#         metrics = extract_metrics(csv_path)
#         rows.append(
#             [video_name, qp, preset] + metrics if metrics else [video_name, qp, preset, None, None, None, None]
#         )
# df = pd.DataFrame(rows, columns=['video_name', 'qp', 'preset', 'elapsed_time', 'bitrate', 'psnr', 'ssim'])

# orig_dir = f'/data/ssd/zhaoy/datasets/{DATASET}/dataset-comp/test/video/yuv'
# rec_dir  = f'/data/ssd/zhaoy/datasets/{DATASET}/compressed/x265/rec'
# msssim_dict = {}  # key = (video_name, qp, preset)

# for rec_file in tqdm(os.listdir(rec_dir)):
#     if not rec_file.endswith('.yuv'):
#         continue
#     video_name, qp, preset = rec_file.replace('.yuv', '').split('_')
#     qp = int(qp.replace('qp', ''))
#     rec_path  = os.path.join(rec_dir, rec_file)
#     orig_path = os.path.join(orig_dir, f'{video_name}.yuv')
#     msssim_val = calc_msssim_yuv(orig_path, rec_path, width=640, height=480)
#     msssim_dict[(video_name, qp, preset)] = msssim_val

# print(msssim_dict)
# df['msssim'] = df.apply(
#     lambda row: msssim_dict.get((row['video_name'], row['qp'], row['preset']), None),
#     axis=1
# )
# display(df)
# df.fillna(1.0)
# df.to_csv('/home/zhaoy/TaCo-Bench/lossy/video/statistics/x265_TouchandGo.csv', index=False)

display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossy/video/statistics/x265_TouchandGo.csv'))

,video_name,qp,preset,elapsed_time,bitrate,psnr,ssim,msssim
0,20220531110519,17,medium,45.01,2953.44,48.655,0.991000,0.996428
1,20220318234954,37,medium,8.10,68.75,38.819,0.967223,0.972882
2,20220410024605,22,medium,26.83,840.31,46.279,0.987200,0.993696
3,20220318234954,47,medium,6.23,23.63,32.670,0.929280,0.933831
4,20220318233930,27,medium,23.98,419.49,43.453,0.980950,0.988289
...,...,...,...,...,...,...,...,...
93,20220531110519,27,medium,26.77,374.25,43.308,0.979620,0.989500
94,20220318234954,42,medium,7.38,40.66,35.762,0.951360,0.957119
95,20220511192421,22,medium,37.03,1122.69,46.271,0.987427,0.994169
96,20220318233930,17,medium,43.43,2514.16,48.907,0.991268,0.996328


In [11]:
''' 2. Object Folder '''
import pandas as pd
DATASET = 'ObjectFolder_1.0'
W, H = 120, 160
# log_dir = f'/data/ssd/zhaoy/datasets/{DATASET}/compressed/x265/log'

# rows = []
# for log in os.listdir(log_dir):
#     if log.endswith('.csv'):
#         video_name, qp, preset = log.replace('.csv', '').split('_')
#         qp = int(qp.replace('qp', ''))
#         csv_path = os.path.join(log_dir, log)
#         metrics = extract_metrics(csv_path)
#         rows.append(
#             [video_name, qp, preset] + metrics if metrics else [video_name, qp, preset, None, None, None, None]
#         )
# df = pd.DataFrame(rows, columns=['video_name', 'qp', 'preset', 'elapsed_time', 'bitrate', 'psnr', 'ssim'])

# orig_dir = f'/data/ssd/zhaoy/datasets/{DATASET}/dataset-comp/test/video/yuv'
# rec_dir  = f'/data/ssd/zhaoy/datasets/{DATASET}/compressed/x265/rec'
# msssim_dict = {}  # key = (video_name, qp, preset)

# for rec_file in tqdm(os.listdir(rec_dir)):
#     if not rec_file.endswith('.yuv'):
#         continue
#     video_name, qp, preset = rec_file.replace('.yuv', '').split('_')
#     qp = int(qp.replace('qp', ''))
#     rec_path  = os.path.join(rec_dir, rec_file)
#     orig_path = os.path.join(orig_dir, f'{video_name}.yuv')
#     msssim_val = calc_msssim_yuv(orig_path, rec_path, width=W, height=H)
#     msssim_dict[(video_name, qp, preset)] = msssim_val

# print(msssim_dict)
# df['msssim'] = df.apply(
#     lambda row: msssim_dict.get((row['video_name'], row['qp'], row['preset']), None),
#     axis=1
# )
# display(df)
# df.fillna(1.0)
# df.to_csv('/home/zhaoy/TaCo-Bench/lossy/video/statistics/x265_ObjectFolder.csv', index=False)

display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossy/video/statistics/x265_ObjectFolder.csv'))

,video_name,qp,preset,elapsed_time,bitrate,psnr,ssim,msssim
0,944,47,medium,8.31,16.83,26.010,0.681494,0.999973
1,95,32,medium,1.77,77.72,36.635,0.967579,0.999998
2,958,17,medium,7.93,520.65,44.121,0.992801,1.000000
3,83,47,medium,1.59,6.87,28.682,0.791126,0.999987
4,84,22,medium,7.25,560.44,40.108,0.983941,0.999999
...,...,...,...,...,...,...,...,...
1395,237,37,medium,2.47,39.11,33.426,0.936865,0.999996
1396,456,32,medium,21.95,136.18,35.026,0.952893,0.999997
1397,388,47,medium,1.65,6.92,28.981,0.818917,0.999988
1398,888,47,medium,2.33,12.19,27.247,0.774905,0.999979


In [13]:
''' 3. SSVTP '''
import pandas as pd
DATASET = 'SSVTP'
width, height = 240, 320
# log_dir = f'/data/ssd/zhaoy/datasets/{DATASET}/compressed/x265/log'

# rows = []
# for log in os.listdir(log_dir):
#     if log.endswith('.csv'):
#         video_name, qp, preset = log.replace('.csv', '').split('_')
#         qp = int(qp.replace('qp', ''))
#         csv_path = os.path.join(log_dir, log)
#         metrics = extract_metrics(csv_path)
#         rows.append(
#             [video_name, qp, preset] + metrics if metrics else [video_name, qp, preset, None, None, None, None]
#         )
# df = pd.DataFrame(rows, columns=['video_name', 'qp', 'preset', 'elapsed_time', 'bitrate', 'psnr', 'ssim'])

# orig_dir = f'/data/ssd/zhaoy/datasets/{DATASET}/dataset-comp/test/video/yuv'
# rec_dir  = f'/data/ssd/zhaoy/datasets/{DATASET}/compressed/x265/rec'
# msssim_dict = {}  # key = (video_name, qp, preset)

# for rec_file in tqdm(os.listdir(rec_dir)):
#     if not rec_file.endswith('.yuv'):
#         continue
#     video_name, qp, preset = rec_file.replace('.yuv', '').split('_')
#     qp = int(qp.replace('qp', ''))
#     rec_path  = os.path.join(rec_dir, rec_file)
#     orig_path = os.path.join(orig_dir, f'{video_name}.yuv')
#     msssim_val = calc_msssim_yuv(orig_path, rec_path, width=width, height=height)
#     msssim_dict[(video_name, qp, preset)] = msssim_val

# print(msssim_dict)
# df['msssim'] = df.apply(
#     lambda row: msssim_dict.get((row['video_name'], row['qp'], row['preset']), None),
#     axis=1
# )
# display(df)
# df.fillna(1.0)
# df.to_csv('/home/zhaoy/TaCo-Bench/lossy/video/statistics/x265_SSVTP.csv', index=False)
display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossy/video/statistics/x265_SSVTP.csv'))

,video_name,qp,preset,elapsed_time,bitrate,psnr,ssim,msssim
0,SSVTP,47,medium,1.58,18.71,36.658,0.964178,0.999997
1,SSVTP,22,medium,8.53,267.63,47.784,0.985767,1.000000
2,SSVTP,32,medium,5.73,70.69,44.349,0.980454,1.000000
3,SSVTP,27,medium,7.13,122.64,45.979,0.982915,1.000000
4,SSVTP,17,medium,9.10,559.66,49.791,0.989564,1.000000
5,SSVTP,37,medium,4.32,46.60,42.407,0.977455,0.999999
6,SSVTP,42,medium,2.63,32.27,39.759,0.972366,0.999999


In [17]:
''' 4. YCB-Slide '''
import pandas as pd
DATASET = 'YCB-Slide'
width, height = 240, 320
# log_dir = f'/data/ssd/zhaoy/datasets/{DATASET}/compressed/x265/log'

# rows = []
# for log in os.listdir(log_dir):
#     if log.endswith('.csv'):
#         video_name = log.split('_qp')[0]
#         qp, preset = log.replace('.csv', '').split('_qp')[1].split('_')
#         # video_name, qp, preset = log.replace('.csv', '').split('_')
#         qp = int(qp)
#         csv_path = os.path.join(log_dir, log)
#         metrics = extract_metrics(csv_path)
        
#         rows.append(
#             [video_name, qp, preset] + metrics if metrics else [video_name, qp, preset, None, None, None, None]
#         )
# df = pd.DataFrame(rows, columns=['video_name', 'qp', 'preset', 'elapsed_time', 'bitrate', 'psnr', 'ssim'])
# display(df)

# orig_dir = f'/data/ssd/zhaoy/datasets/{DATASET}/dataset-comp/test/video/yuv'
# rec_dir  = f'/data/ssd/zhaoy/datasets/{DATASET}/compressed/x265/rec'
# msssim_dict = {}  # key = (video_name, qp, preset)

# for rec_file in tqdm(os.listdir(rec_dir)):
#     if not rec_file.endswith('.yuv'):
#         continue
#     video_name = rec_file.split('_qp')[0]
#     qp, preset = rec_file.replace('.csv', '').split('_qp')[1].split('_')
#     qp = int(qp.replace('qp', ''))
#     rec_path  = os.path.join(rec_dir, rec_file)
#     orig_path = os.path.join(orig_dir, f'{video_name}.yuv')
#     msssim_val = calc_msssim_yuv(orig_path, rec_path, width=width, height=height)
#     msssim_dict[(video_name, qp)] = msssim_val

# print(msssim_dict)
# df['msssim'] = df.apply(
#     lambda row: msssim_dict.get((row['video_name'], row['qp']), None),
#     axis=1
# )
# display(df)
# # df.fillna(1.0)
# df.to_csv('/home/zhaoy/TaCo-Bench/lossy/video/statistics/x265_YCB-Slide.csv', index=False)
display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossy/video/statistics/x265_YCB-Slide.csv'))

,video_name,qp,preset,elapsed_time,bitrate,psnr,ssim,msssim
0,048_hammer,47,medium,12.86,6.05,38.138,0.971892,0.999998
1,048_hammer,22,medium,20.98,79.73,47.863,0.987319,1.000000
2,048_hammer,42,medium,12.82,6.74,40.746,0.977289,0.999999
3,055_baseball,27,medium,15.55,22.56,46.474,0.985838,1.000000
4,055_baseball,42,medium,11.95,7.58,40.142,0.973993,0.999998
5,055_baseball,22,medium,22.26,84.76,47.760,0.987267,1.000000
6,042_adjustable_wrench,42,medium,13.46,7.24,40.219,0.971935,0.999999
7,042_adjustable_wrench,37,medium,13.39,8.77,42.609,0.977019,0.999999
8,005_tomato_soup_can,42,medium,12.23,7.34,40.126,0.972040,0.999999
9,005_tomato_soup_can,32,medium,12.68,11.45,44.759,0.981847,1.000000
